# DiCo-NLI runner (Kaggle): Qwen2.5-7B QLoRA cho EU

Luật go/no-go: PLAN.md mục 2d, "Thí nghiệm: Qwen 7B cho EU". Config: `configs/qwen7b_cls_eu.yaml`.

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. Chạy bằng *Save Version → Save & Run All (Commit)*.

- Ô smoke (0.02 epoch) tải model 7B một lần và in tốc độ train; xem dòng `train_runtime` để ước lượng tổng thời gian.
- GPU 0: 3 seed. GPU 1: OOF 3 fold.

Kết quả: tải `runs.zip` ở tab *Output*, bỏ vào `kaggle/` (gồm `qwen7b_cls_eu/seed*/` và `oof/qwen7b_cls_eu/`).

In [ ]:
REPO_URL = 'https://github.com/danh-tc/DiCo-NLI.git'
!git clone --depth 1 $REPO_URL /kaggle/working/dico-nli
%cd /kaggle/working/dico-nli
!git log -1 --oneline

In [ ]:
!pip install -q -e . peft
# peft mới đòi torchao >= 0.16, Kaggle cài sẵn 0.10 -> gỡ torchao (code không dùng nó)
!pip uninstall -y -q torchao
# --no-deps: không để bitsandbytes kéo torch khác về
!pip install -q --no-deps bitsandbytes
!bash scripts/setup_data.sh
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, bitsandbytes as bnb, peft
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'bnb', bnb.__version__, 'peft', peft.__version__)
assert torch.cuda.is_available(), 'Không có CUDA: bật GPU T4 x2 trong Settings'

In [ ]:
%%bash
# Smoke: 0.02 epoch EU bằng 7B thật (cũng là bước tải model, để hai hàng đợi không tải song song)
set -e
CUDA_VISIBLE_DEVICES=0 PYTHONPATH=src python -m dico.train --config configs/qwen7b_cls_eu.yaml --seed 1 --set name=smoke epochs=0.02 > /kaggle/working/smoke.log 2>&1 || { tail -40 /kaggle/working/smoke.log; exit 1; }
grep -E "precision|trainable|train_runtime|train time" /kaggle/working/smoke.log
nvidia-smi --query-gpu=memory.used --format=csv
rm -rf /kaggle/working/runs/smoke
echo smoke OK

In [ ]:
%%bash
# Hai hàng đợi song song, mỗi GPU một hàng; xong cả hai mới sang ô sau
bash scripts/kaggle_queue.sh 0 train:qwen7b_cls_eu:1 train:qwen7b_cls_eu:2 train:qwen7b_cls_eu:3 &
bash scripts/kaggle_queue.sh 1 oof:qwen7b_cls_eu &
wait
cat /kaggle/working/runs/logs/status.txt

In [ ]:
# Nếu có FAIL: xem 40 dòng cuối log của job đó
!grep FAIL /kaggle/working/runs/logs/status.txt || echo "no failures"
!for f in $(grep FAIL /kaggle/working/runs/logs/status.txt | awk '{print $3}'); do echo "== $f"; tail -40 /kaggle/working/runs/logs/$f.log; done
!for s in 1 2 3; do echo seed$s; tail -4 /kaggle/working/runs/logs/0_train_qwen7b_cls_eu_$s.log; done

In [ ]:
!cd /kaggle/working && zip -qr runs.zip runs -x '*/ckpt/*' '*/model/*' && ls -lh runs.zip